# 02 CIGNN EMNIST

> EMNIST letter exp

In [2]:
#| default_exp data

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:

import os
import pandas as pd
import sys
sys.path.append(os.path.abspath('..'))

import torch
import torchvision
from torchvision.transforms import ToTensor
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Dataset
from torchvision.io import read_image, ImageReadMode
import torch.nn as nn
import torch.optim as optim
from torchsummary import summary


In [ ]:
project="CIGNN_GA_VGG16_bn"
n_epochs = 5
train_split = "/data/datasets/HWR/unipen_curated/split/trn.txt"
val_split = "/data/datasets/HWR/unipen_curated/split/val.txt"
test_split = "/data/datasets/HWR/unipen_curated/split/tst.txt"
data_dir = "/data/datasets/HWR/unipen_curated/curated"
model_save_path = "vgg_unipen_curated.pth"
dev = "cuda"
env = "prod"


In [ ]:
# collect all parameters and initialise wandb project
import wandb

wandb.init(
    project=project,
    config = {
        "n_epochs" : n_epochs,
        "train_split" : train_split,
        "val_split" : val_split,
        "test_split" : test_split,
        "data_dir" : data_dir,
        "model_save_path" : model_save_path,
        "dev" : dev,
        "env" : env
    }
)



In [ ]:
    
from dotenv import dotenv_values

env_cfg = {**dotenv_values(f"./env/{env}.env"), **os.environ}


In [ ]:
def unipen_char_to_code(char: str) -> int: 
    """Convert a single char into the corresponding 0-based index.

    This method respects the fact that UNIPEN (curated) does not have 
    samples for the "\"-symbol. A simple ofsetting -33 for the ASCII
    chars therefore leads to one "extra class", causing indexing errors
    down the line.

    In particular: 
        chars = [chr(i) for i in range(33, 123) if not i == 92]
        codes = [unipen_char_to_code(c) for c in chars]
        chars2 = [unipen_code_to_char(cc) for cc in codes]
        chars == chars2


    Args:
        char (str): The Char "!" - "z" to index.

    Returns:
        int: 0-based index for the input char, skipping the ascii code 92.
    """

    print(char)
    res = ord(char) - 33 
    return res - 1 if (res > (92 - 33)) else res 

def unipen_code_to_char(code: int) -> str: 
    """Convert a 0-based index int to the corresponding ASCII char in the context of the unipen curated dataset.

    This method avoids indexing errors that can happen because the unipen dataset does not provide samples for ascii char 
    92, so simple +-33 conversions cause one empty extra index.

    In particular: 
        chars = [chr(i) for i in range(33, 123) if not i == 92]
        codes = [unipen_char_to_code(c) for c in chars]
        chars2 = [unipen_code_to_char(cc) for cc in codes]
        chars == chars2

    Args:
        code (int): integer of the index to convert to char.

    Returns:
        str: The char after conversion. 
    """

    code = code if code < (92 - 33) else code + 1
    return chr(33 + code)




In [ ]:

# load data from curated dataset 
class UnipenCuratedDataset(Dataset):
    def __init__(self, annotations_file, img_dir, transform=None, target_transform=None):
        self.device = torch.device(dev)
        
        self.img_labels = []
        with open(annotations_file, "r") as fh:
            self._img_labels = [[line.strip(), line.strip().split("/")[0]] for line in fh.readlines()]
            self.img_labels = pd.DataFrame(self._img_labels)
        self.img_dir = img_dir
        self.transform = transform
        self.target_transform = target_transform

    def __len__(self):
        return len(self.img_labels)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.img_labels.iloc[idx, 0])
        image = read_image(img_path, mode=ImageReadMode.RGB)
        label = self.img_labels.iloc[idx, 1]
        if self.transform:
            image = self.transform(image)
        if self.target_transform:
            label = self.target_transform(label)
        return image.to(torch.float32).to(self.device), int(label) - 33 if int(label) <= 92 else int(label) - 34


In [ ]:
def generate_dataset(config=None):
    def transform(img):
        return img

    train_data = UnipenCuratedDataset(train_split, data_dir, transform=transform)
    train_dataloader = DataLoader(dataset=train_data, batch_size=16, shuffle=True)
    val_data = UnipenCuratedDataset(val_split, data_dir, transform=transform)
    val_dataloader = DataLoader(dataset=val_data, batch_size=16, shuffle=True) 
    
    return train_dataloader, val_dataloader

In [ ]:
# Load the EMNIST dataset (one test and one training set, using the "balanced" or "letters" split for example)

test_data = UnipenCuratedDataset(test_split, data_dir)
test_dataloader = DataLoader(dataset=test_data, batch_size=16, shuffle=True)

In [ ]:


# the correct #numclass is 93, however, the curated unipen dataset does not have samples for ascii 92 ("\"-symbol")
vgg_bn = torchvision.models.vgg16_bn(num_classes=93)
device = torch.device(dev)
print(f"device: {device}")
vgg_bn.to(device)
sample, labels = next(iter(test_dataloader))


In [ ]:
model = vgg_bn
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)


In [ ]:
# Training loop with validation and checkpoint saving
best_val_loss = float('inf')  # Initialize the best validation loss
checkpoint_path = 'best_model_checkpoint.pth'  # Path to save the best model
train_dataloader, val_dataloader = generate_dataset()

for epoch in range(n_epochs):
    model.train()
    running_loss = 0.0

    # Training phase
    for batch_idx, (images, labels) in enumerate(train_dataloader, start=1):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()  # Clear gradients
        outputs = model(images)  # Forward pass
        loss = criterion(outputs, labels)  # Calculate loss
        loss.backward()  # Backward pass
        optimizer.step()  # Update weights
        running_loss += loss.item()

        # Print progress every 10 batches
        if batch_idx % 10 == 0:
            wandb.log({"train_loss": loss.item()})
            print(f"Epoch [{epoch+1}/{n_epochs}], Batch [{batch_idx}/{len(train_dataloader)}], Loss: {running_loss/batch_idx:.4f}")

    avg_train_loss = running_loss / len(train_dataloader)
    print(f"Epoch [{epoch+1}/{n_epochs}], Training Loss: {avg_train_loss:.4f}")

    # Validation phase
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for images, labels in val_dataloader:  # Use a separate validation dataloader
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)  # Forward pass
            loss = criterion(outputs, labels)  # Calculate validation loss
            val_loss += loss.item()

    avg_val_loss = val_loss / len(val_dataloader)
    print(f"Epoch [{epoch+1}/{n_epochs}], Validation Loss: {avg_val_loss:.4f}")
    wandb.log({"val_loss": loss.item()})


    # Checkpoint saving
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), checkpoint_path)
        print(f"Best model updated and saved at epoch {epoch+1} with Validation Loss: {avg_val_loss:.4f}")

# Testing loop
model.load_state_dict(torch.load(checkpoint_path))  # Load the best model checkpoint
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_dataloader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)  # Forward pass
        predicted = outputs.argmax(dim=1)  # Get predicted class
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

accuracy = 100 * correct / total
print(f"Test Accuracy: {accuracy:.2f}%")
wandb.log({"test_acc": accuracy})


In [ ]:
import matplotlib.pyplot as plt
import torch

# Function to display an image and its labels
def show_sample(image, true_label, predicted_label):

    # The .permute call is required to transpose the image. We receive a shape (3, 64, 64) here
    # but .imshow() expects a shape with the channels at the end, like (64, 64, 3)
    image = image.permute(1, 2, 0)
    image = image.cpu().squeeze().numpy()  # Convert to 2D array for display
    plt.imshow(image)
    print(true_label)
    print(predicted_label)
    plt.title(f"True: {unipen_code_to_char(true_label)}, Predicted: {unipen_code_to_char(predicted_label)}")  
    plt.axis('off')
    plt.show()

# Run inference and display samples
model.eval()
num_samples_to_show = 5
shown_samples = 0

with torch.no_grad():
    for images, labels in test_dataloader:
        outputs = model(images)
        print(outputs)
        print(outputs.shape)
        probs, predicted = torch.max(outputs, 1)

        for i in range(len(images)):
            if shown_samples >= num_samples_to_show:
                break
            true_label = labels[i].item()
            predicted_label = predicted[i].item()
            # Shift by -1 to match a=1, ..., z=26 in EMNIST "letters"
            show_sample(images[i], true_label, predicted_label )
            shown_samples += 1
        
        if shown_samples >= num_samples_to_show:
            break

In [16]:
from torch import nn

# Get the activation at the penultimate layer

# Interestingly, this is the final non-flat layer 512x7x7
# when I would have expected it to be the final dense layer
# right before the Softmax (4094 -> #numclass)
embedding_model = nn.Sequential(*list(model.children())[:-1])

# Run inference and display samples
embedding_model.eval()
num_samples_to_show = 5
shown_samples = 0

with torch.no_grad():
    for images, labels in test_dataloader:
        outputs = embedding_model(images)
        
        print(outputs.shape)
        break
        

torch.Size([16, 512, 7, 7])


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()